In [ ]:
"""
Flash Attention
===============
This is a Helion implementation of the Flash Attention v2 algorithm.
Author: Cody Wang
"""

import helion
import helion.language as hl
import torch
from torch import Tensor

In [2]:
from triton.testing import do_bench
def test_kernel(kernel_fn, spec_fn, *args):
    """Test a Helion kernel against a reference implementation."""
    # Run our implementation
    result = kernel_fn(*args)
    # Run reference implementation
    expected = spec_fn(*args)

    # Check if results match
    torch.testing.assert_close(result, expected)
    print("✅ Results Match ✅")

def benchmark_kernel(kernel_fn, *args, **kwargs):
    """Benchmark a Helion kernel."""
    no_args = lambda: kernel_fn(*args, **kwargs)
    time_in_ms = do_bench(no_args)
    print(f"⏱ Time: {time_in_ms} ms")

def compare_implementations(kernel_fn, spec_fn, *args, **kwargs):
    """Benchmark a Helion kernel and its reference implementation."""
    kernel_no_args = lambda: kernel_fn(*args, **kwargs)
    spec_no_args = lambda: spec_fn(*args, **kwargs)
    kernel_time = do_bench(kernel_no_args)
    spec_time = do_bench(spec_no_args)
    print(f"⏱ Helion Kernel Time: {kernel_time:.3f} ms, PyTorch Reference Time: {spec_time:.3f} ms, Speedup: {spec_time/kernel_time:.3f}x")

In [ ]:
@helion.kernel(
    autotune_effort="none", # Autotuning disabled for development
    dot_precision="ieee"    # Using ieee precision to match spec
)
def flashatt_fwd(q: Tensor, k: Tensor, v: Tensor) -> Tensor:
    seq_len, d_head = q.size()
    qk_scale = 1 / (d_head ** 0.5)

    out = torch.zeros_like(q, dtype=q.dtype, device=q.device)

    for tile_q in hl.tile(seq_len):
        q_tile = q[tile_q, :]

        l_j = hl.zeros([tile_q], dtype=q.dtype, device=q.device)
        m_i = hl.full([tile_q], -float("inf"), dtype=q.dtype, device=q.device)
        o_j = hl.zeros([tile_q, d_head])

        for tile_kv in hl.tile(seq_len):
            k_tile = k[tile_kv, :]
            v_tile = v[tile_kv, :]

            qk = q_tile @ k_tile.T * qk_scale

            m_ij = torch.maximum(m_i, torch.amax(qk, -1))
            p = torch.exp(qk - m_ij[:, None])
            scale_factor = torch.exp(m_i - m_ij)
            l_j = scale_factor * l_j + torch.sum(p, -1)

            o_j = o_j * scale_factor[:, None] + p @ v_tile

            m_i = m_ij

        out[tile_q, :] = o_j / l_j[:, None]

    return out

# Test the kernel
"""
Arguments:
    q: (batch_size, seqlen_q, nheads, d)
    k: (batch_size, seqlen_k, nheads_k, d)
    v: (batch_size, seqlen_k, nheads_k, d)
"""
size = (128, 128)
q = torch.randn(size, device="cuda")
k = torch.randn(size, device="cuda")
v = torch.randn(size, device="cuda")

flashatt_spec = torch.nn.functional.scaled_dot_product_attention
test_kernel(flashatt_fwd, flashatt_spec, q, k, v)

Testing PyTorch implementation:
✅ Results Match ✅
Testing Helion implementation:


Using default config: @helion.kernel(config=helion.Config(block_sizes=[32, 32], indexing=['pointer', 'pointer', 'pointer', 'pointer'], load_eviction_policies=['', '', ''], num_stages=1, num_warps=4, pid_type='flat', range_flattens=[None, None], range_multi_buffers=[None, None], range_num_stages=[0, 0], range_unroll_factors=[0, 0], range_warp_specializes=[]), static_shapes=True)


✅ Results Match ✅
